# 13. 치환 라이브러리 확장 (2차)

## 이번 노트북에서 할 것
- 09에서 확인한 빈도분석 상위 규칙(Aliphatic_long_chain, Oxygen-nitrogen_single_bond,
  isolated_alkene, imine_1, Sulfonic_acid_2 등) 중 문헌 근거가 명확한 것 선별
- SMARTS 패턴 설계 (고리 인접 여부 먼저 판단 → Case A/B 적용 여부 결정)
- replacement_library.py 확장 및 회귀 테스트
- (여유 시간에) held-out set으로 확장된 라이브러리의 커버리지 재측정

## 간략한 정리 (12까지)
- 도구/에이전트 계층 완성, 멀티 LLM 지원(Gemini+Qwen), Case A/B 재조립 로직 완성
- Held-out 평가: 단일문제 94% 개선(58% 완전해결+36% 부분), 다중문제 7/7 완전해결
- 3단계 검증체계 완성: 구조규칙(FilterCatalog) → Tox21(12-assay) → Ames(변이원성)
  167개 표본 검증: Tox21 p=0.00005, Ames p<0.00001, 둘 다 유의, Ames 효과크기 10배 큼
- 현재 라이브러리: 9개 규칙 (문헌기반 7 + MMPA데이터기반 2)
- 마감까지 약 14일 남음(7/24→8/7 16:00) → 코드 확장 + 문헌·제안서 병행 전략으로 진행

## 다음에 해야 할 것 (오늘 끝나면)
- 확장된 라이브러리로 held-out 커버리지 재측정 (현재 166/1174, 14%)
- 제안서 hwpx 양식 작업 착수
- 다이어그램(도구계층+에이전트계층 구조도) 제작

In [1]:
# 셀 1
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 36.2 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 134 (delta 57), reused 93 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 324.62 KiB | 6.76 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "Dec32th"
!git config --global user.name "hyekyeong.w@gmail.com"

In [4]:
# 셀 4
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()
print("도구 로드 확인 완료")

[09:06:10] WARNING: not removing hydrogen atom without neighbors
[09:06:11] Explicit valence for atom # 8 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 3 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 9 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 5 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 16 Al, 6, is greater than permitted
[09:06:11] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[09:06:11] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [5]:
from collections import Counter

rule_counter = Counter()
for s in data['smiles_train'][:1000]:
    for p in detect_toxicophores(s):
        rule_counter[p['rule_name']] += 1

covered = set(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리 커버 규칙:", covered)
print()
for rule, count in rule_counter.most_common(20):
    status = "✅ 커버됨" if rule in covered else "❌ 미커버"
    print(f"{rule}: {count}회  {status}")

현재 라이브러리 커버 규칙: {'thiourea', 'aniline', 'alkyl_halide', 'amide', 'acyl_halide', 'michael_acceptor', 'phenol', 'nitro_group', 'aldehyde'}

Aliphatic_long_chain: 133회  ❌ 미커버
Oxygen-nitrogen_single_bond: 70회  ❌ 미커버
isolated_alkene: 59회  ❌ 미커버
nitro_group: 44회  ✅ 커버됨
alkyl_halide: 44회  ✅ 커버됨
aniline: 43회  ✅ 커버됨
Sulfonic_acid_2: 36회  ❌ 미커버
imine_1: 31회  ❌ 미커버
Michael_acceptor_1: 30회  ❌ 미커버
aldehyde: 28회  ✅ 커버됨
phosphor: 21회  ❌ 미커버
quaternary_nitrogen_1: 17회  ❌ 미커버
beta-keto/anhydride: 17회  ❌ 미커버
quaternary_nitrogen_2: 16회  ❌ 미커버
thiol_2: 12회  ❌ 미커버
imine_2: 12회  ❌ 미커버
catechol: 12회  ❌ 미커버
halogenated_ring_1: 11회  ❌ 미커버
heavy_metal: 11회  ❌ 미커버
het-C-het_not_in_ring: 11회  ❌ 미커버


In [8]:
target_names = ["Sulfonic_acid_2", "imine_1"]

examples = {name: None for name in target_names}
for s in data['smiles_train'][:1000]:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in target_names and examples[p['rule_name']] is None:
            examples[p['rule_name']] = (s, p['atom_indices'])

for name, info in examples.items():
    print(f"{name}: {info}")

Sulfonic_acid_2: ('Nc1ccc2cc(S(=O)(=O)O)cc(O)c2c1', [7, 8, 9, 10])
imine_1: ('CSC(C)(C)/C=N\\O', [5, 6])


In [9]:
def show_atoms(smiles, atom_indices):
    mol = Chem.MolFromSmiles(smiles)
    for idx in atom_indices:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  인덱스 {idx}: {atom.GetSymbol()} (이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")

print("Sulfonic_acid_2:")
show_atoms('Nc1ccc2cc(S(=O)(=O)O)cc(O)c2c1', [7, 8, 9, 10])

print("\nimine_1:")
show_atoms('CSC(C)(C)/C=N\\O', [5, 6])

Sulfonic_acid_2:
  인덱스 7: S (이웃: ['C', 'O', 'O', 'O'])
  인덱스 8: O (이웃: ['S'])
  인덱스 9: O (이웃: ['S'])
  인덱스 10: O (이웃: ['S'])

imine_1:
  인덱스 5: C (이웃: ['C', 'N'])
  인덱스 6: N (이웃: ['C', 'O'])


In [10]:
!cat src/tools/replacement_library.py


REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
   

In [11]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "sulfonic_acid": {
        "problem_smarts": "S(=O)(=O)[OX2H1]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
        ],
    },
    "oxime": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [12]:
import importlib
import src.tools.replacement_library
importlib.reload(src.tools.replacement_library)
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

# sulfonic_acid 테스트
sulfonic_test = "Nc1ccc2cc(S(=O)(=O)O)cc(O)c2c1"
print("sulfonic_acid core_test:", find_core_and_target(sulfonic_test, "sulfonic_acid"))
print("sulfonic_acid fix_test:", propose_fix(sulfonic_test, "sulfonic_acid", candidate_idx=0))

# oxime 테스트
oxime_test = "CSC(C)(C)/C=N\\O"
print("\noxime core_test:", find_core_and_target(oxime_test, "oxime"))
print("oxime fix_test:", propose_fix(oxime_test, "oxime", candidate_idx=0))

sulfonic_acid core_test: {'core': 'Nc1ccc2cc([*:1])cc(O)c2c1', 'target_removed': 'O=S(=O)(O)[*:1]'}
sulfonic_acid fix_test: {'new_smiles': 'Nc1ccc2cc(S(N)(=O)=O)cc(O)c2c1', 'candidate_used': 'sulfonamide', 'rationale': '생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 중성에 가까워 약물유사성이 개선됨', 'is_valid': True}

oxime core_test: {'core': 'CSC(C)(C)[*:1]', 'target_removed': 'O/N=C\\[*:1]'}
oxime fix_test: {'new_smiles': 'CSC(C)(C)CN', 'candidate_used': 'amine (reduced)', 'rationale': '옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 경로를 제거함', 'is_valid': True}


In [13]:
print("\n=== 회귀 테스트 ===")
print("alkyl_halide:", propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print("phenol:", propose_fix("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol", candidate_idx=0))
print("amide:", propose_fix("CNC(=O)c1ccccc1", "amide", candidate_idx=0))


=== 회귀 테스트 ===
alkyl_halide: {'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
phenol: {'new_smiles': 'O=C(c1ccc(Cl)cc1)c1ccc(Cl)cc1', 'candidate_used': 'chlorine', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 (quinone 형성 등) 경로를 차단하는 것으로 추정', 'is_valid': True}
amide: {'new_smiles': 'NC(=O)Nc1ccccc1', 'candidate_used': 'urea', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 계열(우레아 활용)로 일관성 있음', 'is_valid': True}


In [14]:
!git add src/tools/replacement_library.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/replacement_library.py



In [15]:
!git commit -m "Expand replacement library: add sulfonic_acid (->sulfonamide) and oxime (->reduced amine), both validated with regression tests passing"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 873befe] Expand replacement library: add sulfonic_acid (->sulfonamide) and oxime (->reduced amine), both validated with regression tests passing
 1 file changed, 19 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1000 bytes | 1000.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   d5beb2b..873befe  main -> main


In [16]:
count_known_v2 = 0
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_known_v2 += 1

print(f"이전(9개 규칙): 166개 / 이번(11개 규칙): {count_known_v2}개")

이전(9개 규칙): 166개 / 이번(11개 규칙): 166개


In [17]:
# 새 규칙이 실제로 test set에 있는지 직접 확인
count_sulfonic = 0
count_oxime = 0

for s in data['smiles_test']:
    p = detect_toxicophores(s)
    rule_names = [x['rule_name'] for x in p]
    if 'sulfonic_acid' in rule_names:
        count_sulfonic += 1
    if 'oxime' in rule_names:
        count_oxime += 1

print(f"sulfonic_acid 발견: {count_sulfonic}개")
print(f"oxime 발견: {count_oxime}개")

sulfonic_acid 발견: 0개
oxime 발견: 0개


In [18]:
print(get_replacement_candidates("Sulfonic_acid_2"))
print(get_replacement_candidates("imine_1"))

None
None


In [19]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [20]:
importlib.reload(src.tools.replacement_library)
from src.tools.replacement_library import get_replacement_candidates

print(get_replacement_candidates("Sulfonic_acid_2"))
print(get_replacement_candidates("imine_1"))

count_known_v3 = 0
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_known_v3 += 1

print(f"\n이전(9개 규칙): 166개 / 이번(11개 규칙, 이름 수정): {count_known_v3}개")

{'problem_smarts': 'S(=O)(=O)[OX2H1]', 'candidates': [{'smiles': 'S(=O)(=O)N', 'name': 'sulfonamide', 'rationale': '생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 중성에 가까워 약물유사성이 개선됨'}]}
{'problem_smarts': 'C=N[OX2H1]', 'candidates': [{'smiles': 'CN', 'name': 'amine (reduced)', 'rationale': '옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 경로를 제거함'}]}

이전(9개 규칙): 166개 / 이번(11개 규칙, 이름 수정): 227개


In [21]:
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

sulfonic_test = "Nc1ccc2cc(S(=O)(=O)O)cc(O)c2c1"
print("sulfonic fix (새 키명):", propose_fix(sulfonic_test, "Sulfonic_acid_2", candidate_idx=0))

oxime_test = "CSC(C)(C)/C=N\\O"
print("oxime fix (새 키명):", propose_fix(oxime_test, "imine_1", candidate_idx=0))

sulfonic fix (새 키명): {'new_smiles': 'Nc1ccc2cc(S(N)(=O)=O)cc(O)c2c1', 'candidate_used': 'sulfonamide', 'rationale': '생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 중성에 가까워 약물유사성이 개선됨', 'is_valid': True}
oxime fix (새 키명): {'new_smiles': 'CSC(C)(C)CN', 'candidate_used': 'amine (reduced)', 'rationale': '옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 경로를 제거함', 'is_valid': True}


In [22]:
!git add src/tools/replacement_library.py
!git commit -m "Fix critical naming bug: replacement_library keys must match FilterCatalog's actual rule names (Sulfonic_acid_2, imine_1), not arbitrary names; coverage increased 166->227 (+37%)"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 9a9e483] Fix critical naming bug: replacement_library keys must match FilterCatalog's actual rule names (Sulfonic_acid_2, imine_1), not arbitrary names; coverage increased 166->227 (+37%)
 1 file changed, 2 insertions(+), 2 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 584 bytes | 584.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   873befe..9a9e483  main -> main


In [23]:
!pip install openai -q

In [24]:
from openai import OpenAI

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
)
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [25]:
multi_test_v2 = None
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if len(known) >= 2:
        rule_names = [x['rule_name'] for x in known]
        if 'Sulfonic_acid_2' in rule_names or 'imine_1' in rule_names:
            multi_test_v2 = s
            break

print("찾은 분자:", multi_test_v2)
if multi_test_v2:
    print("문제들:", detect_toxicophores(multi_test_v2))

찾은 분자: Nc1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O
문제들: [{'rule_name': 'quinone_A(370)', 'atom_indices': [10, 11, 12, 13, 14, 19, 20, 21]}, {'rule_name': 'anthranil_one_A(38)', 'atom_indices': [0, 1, 2, 7, 8, 10, 11, 12, 13, 14, 15]}, {'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 7, 8, 10, 11]}, {'rule_name': 'Sulfonic_acid_2', 'atom_indices': [3, 4, 5, 6]}]


In [26]:
if multi_test_v2:
    result_v2 = iterative_fix_loop(
        multi_test_v2, max_iterations=10,
        llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
    )
    print("\n상태:", result_v2['status'])
    for h in result_v2['history']:
        print(h)
else:
    print("해당 분자를 찾지 못했습니다 - 검색 범위를 늘려야 할 수 있습니다.")


상태: stuck
{'step': 0, 'smiles': 'Nc1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O', 'problems': [{'rule_name': 'quinone_A(370)', 'atom_indices': [10, 11, 12, 13, 14, 19, 20, 21]}, {'rule_name': 'anthranil_one_A(38)', 'atom_indices': [0, 1, 2, 7, 8, 10, 11, 12, 13, 14, 15]}, {'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 7, 8, 10, 11]}, {'rule_name': 'Sulfonic_acid_2', 'atom_indices': [3, 4, 5, 6]}]}
{'step': 1, 'smiles': 'NC(=O)c1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O', 'fixed_rule': 'aniline', 'problem_reason': '방향족 아민은 대사 활성화로 인한 유전독성 우려가 명확하고 아마이드화나 치환으로 완화가 가능한 반면, 술폰산기는 이온화·용해도·결합 특성에 미치는 영향이 커서 우선적으로 변경하기 어렵기 때문입니다.', 'candidate_used': 'acetamide (acylated amine)', 'candidate_reason': '1차 방향족 아민을 아마이드로 아실화하면 N-hydroxylation에 의한 반응성 대사체 생성을 직접 차단하면서 기존 아릴 아민 골격의 결합·전자 특성을 비교적 유지할 수 있습니다.', 'problems': [{'rule_name': 'quinone_A(370)', 'atom_indices': [12, 13, 14, 15, 16, 21, 22, 23]}, {'rule_name': 'Sulfonic_acid_2', 'atom_indices': [5, 6, 7, 8]}]}


In [27]:
current_smiles = 'NC(=O)c1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O'
result_check = propose_fix(current_smiles, "Sulfonic_acid_2", candidate_idx=0)
print(result_check)

core_check = find_core_and_target(current_smiles, "Sulfonic_acid_2")
print(core_check)

None
None


In [28]:
mol_check = Chem.MolFromSmiles(current_smiles)
pattern_neutral = Chem.MolFromSmarts("S(=O)(=O)[OX2H1]")
pattern_ionized = Chem.MolFromSmarts("S(=O)(=O)[O-]")

print("중성형 패턴 매치:", mol_check.HasSubstructMatch(pattern_neutral))
print("이온형 패턴 매치:", mol_check.HasSubstructMatch(pattern_ionized))

중성형 패턴 매치: False
이온형 패턴 매치: True


In [29]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [30]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix, iterative_fix_loop

# 이온형 재확인
print(find_core_and_target(current_smiles, "Sulfonic_acid_2"))

# 회귀 테스트 (중성형도 여전히 되는지)
print(propose_fix(sulfonic_test, "Sulfonic_acid_2", candidate_idx=0))

{'core': 'NC(=O)c1c([*:1])cc(Br)c2c1C(=O)c1ccccc1C2=O', 'target_removed': 'O=S(=O)([O-])[*:1]'}
{'new_smiles': 'Nc1ccc2cc(S(N)(=O)=O)cc(O)c2c1', 'candidate_used': 'sulfonamide', 'rationale': '생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 중성에 가까워 약물유사성이 개선됨', 'is_valid': True}


In [31]:
result_v3 = iterative_fix_loop(
    multi_test_v2, max_iterations=10,
    llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
)
print("상태:", result_v3['status'])
for h in result_v3['history']:
    print(h)

상태: no_known_fix
{'step': 0, 'smiles': 'Nc1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O', 'problems': [{'rule_name': 'quinone_A(370)', 'atom_indices': [10, 11, 12, 13, 14, 19, 20, 21]}, {'rule_name': 'anthranil_one_A(38)', 'atom_indices': [0, 1, 2, 7, 8, 10, 11, 12, 13, 14, 15]}, {'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 7, 8, 10, 11]}, {'rule_name': 'Sulfonic_acid_2', 'atom_indices': [3, 4, 5, 6]}]}
{'step': 1, 'smiles': 'NC(=O)c1c(S(=O)(=O)[O-])cc(Br)c2c1C(=O)c1ccccc1C2=O', 'fixed_rule': 'aniline', 'problem_reason': '아닐린은 대사적으로 활성화되어 유전독성을 일으킬 수 있는 1차 방향족 아민이며 말단 치환기로 변형이 비교적 용이하므로, 설폰산기보다 먼저 독성 저감을 시도하는 것이 타당하다.', 'candidate_used': 'acetamide (acylated amine)', 'candidate_reason': '아세트아마이드로 아실화하면 1차 방향족 아민의 N-수산화 대사를 차단하면서도 기존 아릴 치환 패턴과 결합 특성을 더 잘 유지할 수 있다.', 'problems': [{'rule_name': 'quinone_A(370)', 'atom_indices': [12, 13, 14, 15, 16, 21, 22, 23]}, {'rule_name': 'Sulfonic_acid_2', 'atom_indices': [5, 6, 7, 8]}]}
{'step': 2, 'smiles': 'NC(=O)c1c(S(N)(=O)=O)cc(Br)c2c1C(=O

In [32]:
!git add src/tools/replacement_library.py
!git commit -m "Fix Sulfonic_acid_2 SMARTS to include ionized form ([OX2H1,OX1-]); verified full 3-step resolution (aniline->acetamide, Sulfonic_acid_2->sulfonamide, quinone_A honestly reported as no_known_fix)"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 30ef15f] Fix Sulfonic_acid_2 SMARTS to include ionized form ([OX2H1,OX1-]); verified full 3-step resolution (aniline->acetamide, Sulfonic_acid_2->sulfonamide, quinone_A honestly reported as no_known_fix)
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 569 bytes | 569.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   9a9e483..30ef15f  main -> main
